In [1]:

# Load the FAQ data and build the search index
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
from types import SimpleNamespace

load_dotenv(dotenv_path=Path.cwd() / ".env")

if not os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY is not set; using a local fallback response for the notebook demo.")

cwd = Path.cwd()
for candidate in [cwd, cwd / "module5", cwd.parent / "module5"]:
    if (candidate / "rag_helper.py").exists():
        sys.path.insert(0, str(candidate))
        break

from ingest import load_faq_data, build_index
from rag_helper import RAGBase

documents = load_faq_data()
index = build_index(documents)
print(f"Loaded {len(documents)} documents into the search index.")


OPENAI_API_KEY is not set; using a local fallback response for the notebook demo.
Loaded 1380 documents into the search index.


In [2]:

# Configure tracing for the RAG workflow
from opentelemetry import trace
from opentelemetry.trace import ProxyTracerProvider
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor, SpanExporter, SpanExportResult

class RecordingSpanExporter(SpanExporter):
    def __init__(self):
        self.spans = []

    def export(self, spans):
        self.spans.extend(spans)
        return SpanExportResult.SUCCESS

    def shutdown(self):
        pass

    def force_flush(self):
        return True


def calculate_cost(model, usage):
    cost = 0.0
    if "gpt-5.4-mini" in model:
        cost = (usage.input_tokens * 0.15 + usage.output_tokens * 0.60) / 1_000_000
    return cost


class LocalLLMClient:
    class _Completions:
        def create(self, model, messages):
            return SimpleNamespace(
                usage=SimpleNamespace(input_tokens=700, output_tokens=120),
                choices=[SimpleNamespace(message=SimpleNamespace(content="This is a local fallback answer for the homework demo."))],
            )

    class _Chat:
        def __init__(self, outer):
            self.completions = outer._Completions()

    def __init__(self):
        self.chat = self._Chat(self)


class RAGTraced(RAGBase):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.tracer = trace.get_tracer("llm-zoomcamp")

    def search(self, query, num_results=5):
        with self.tracer.start_as_current_span("search") as span:
            results = super().search(query, num_results=num_results)
            span.set_attribute("num_results", len(results))
            return results

    def llm(self, prompt):
        with self.tracer.start_as_current_span("llm") as span:
            response = self.llm_client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": self.instructions},
                    {"role": "user", "content": prompt},
                ],
            )
            usage = getattr(response, "usage", None)
            input_tokens = getattr(usage, "input_tokens", None)
            output_tokens = getattr(usage, "output_tokens", None)
            if input_tokens is None:
                input_tokens = getattr(usage, "prompt_tokens", None)
            if output_tokens is None:
                output_tokens = getattr(usage, "completion_tokens", None)
            cost = calculate_cost(self.model, type("Usage", (), {"input_tokens": input_tokens, "output_tokens": output_tokens})())
            span.set_attribute("input_tokens", input_tokens or 0)
            span.set_attribute("output_tokens", output_tokens or 0)
            span.set_attribute("cost", cost)
            return response.choices[0].message.content

    def rag(self, query):
        with self.tracer.start_as_current_span("rag") as span:
            search_results = self.search(query)
            prompt = self.build_prompt(query, search_results)
            answer = self.llm(prompt)
            span.set_attribute("query", query)
            return answer


provider = TracerProvider()
recording_exporter = RecordingSpanExporter()
provider.add_span_processor(SimpleSpanProcessor(recording_exporter))
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
if isinstance(trace.get_tracer_provider(), ProxyTracerProvider):
    trace.set_tracer_provider(provider)

llm_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY")) if os.getenv("OPENAI_API_KEY") else LocalLLMClient()
rag = RAGTraced(index=index, llm_client=llm_client, model="gpt-5.4-mini")
print("Tracing is configured.")


Tracing is configured.


In [3]:

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print("ANSWER\n")
print(answer)
print("\nSPAN NAMES:")
for span in recording_exporter.spans:
    print(span.name)


ANSWER

This is a local fallback answer for the homework demo.

SPAN NAMES:
search
llm
rag


In [4]:

span_names = [span.name for span in recording_exporter.spans]
print("Number of spans:", len(span_names))
print("Span names:", span_names)


Number of spans: 3
Span names: ['search', 'llm', 'rag']


In [5]:

llm_span = next(span for span in recording_exporter.spans if span.name == "llm")
attrs = dict(llm_span.attributes or {})
print("Attributes on llm span:")
print(attrs)


Attributes on llm span:
{'input_tokens': 700, 'output_tokens': 120, 'cost': 0.000177}


In [6]:

search_span = next(span for span in recording_exporter.spans if span.name == "search")
llm_span = next(span for span in recording_exporter.spans if span.name == "llm")
search_ms = (search_span.end_time - search_span.start_time) / 1_000_000
llm_ms = (llm_span.end_time - llm_span.start_time) / 1_000_000
print(f"Search span duration: {search_ms:.2f} ms")
print(f"LLM span duration: {llm_ms:.2f} ms")


Search span duration: 6.89 ms
LLM span duration: 0.05 ms


In [7]:

import sqlite3

class SQLiteSpanExporter(SpanExporter):
    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

provider = trace.get_tracer_provider()
provider.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter("traces.db")))

rag.rag(query)

conn = sqlite3.connect("traces.db")
print(pd.read_sql_query("SELECT name FROM spans", conn))
conn.close()


     name
0  search
1     llm
2     rag
3  search
4     llm
5     rag
6  search
7     llm
8     rag


In [8]:

conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("SELECT name, start_time, end_time FROM spans WHERE name != 'rag'", conn)
conn.close()

df["duration_ms"] = (df["end_time"] - df["start_time"]) / 1_000_000
print(df.groupby("name")["duration_ms"].sum().sort_values(ascending=False))


name
search    29.697447
llm        0.488833
Name: duration_ms, dtype: float64


In [9]:

for _ in range(3):
    rag.rag(query)

conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("SELECT name, input_tokens FROM spans WHERE name = 'llm'", conn)
conn.close()

print(df)
print("Input token variation:", df["input_tokens"].tolist())


  name  input_tokens
0  llm           700
1  llm           700
2  llm           700
3  llm           700
4  llm           700
5  llm           700
Input token variation: [700, 700, 700, 700, 700, 700]
